# B3U — Unified Multiplicity (UM) and H1b-K3

## Why
B3S closed the verification line: an expanded SAN vocabulary recognises background masks (S1 fixed when ungated), but verification only subtracts, so it needs an image-level regime gate — and no GT-free signal separates the regimes. The Fundamental Review proposes two candidates:

- **UM (unified multiplicity):** estimate how many objects each mask holds — 0, 1 or k — with one local, two-sided rule, so no image gate is needed.
- **H1b-K3:** H1b with K3 (depth-gradient watershed) on whole merged regions where H1b's own gate asks for geometric multiplicity.

## Pre-declared definitions (before any B3U result)

**UM** on GPV_R4's kept masks:
- target area **aᵢ** = pixels of mask i where the target wins over every vocabulary class (tiled expanded-vocabulary map, B3S Step 5);
- **verified single masks** = target-dominant, not rule-A merged, SAM2 score ≥ the image's median mask score;
- object size **s** = median aᵢ of verified single masks; fallback if fewer than 3: median aᵢ of all target-dominant masks; if none: median aᵢ of all masks (≥ 1 px);
- **mᵢ = round(aᵢ / s)**; count = Σ mᵢ.

**H1b-K3:**
- anchor, rule A and H1b's SG two-stage gate exactly as in H1b_v1;
- gate OFF → H1b_v1's count;
- gate ON → anchor masks + Σ over qualifying merged regions of max(1, K3 segments), with GPV_K's region definition (merged mask minus anchor pixels and earlier regions; qualifies if area ≥ 0.5 × A_ref; A_ref = median anchor-mask area; K3 segments ≥ 0.25 × A_ref).

**Targets** (train_dev400, paired bootstrap 10,000, within-stratum seeds 1000+): **T1** S1, S2 not significantly worse than B0; **T2′** S5, S6 not significantly worse than H1b_v1; **T3′** stratum-reweighted MAE significantly better than H1b_v1. A candidate passing all three goes to a single held-out comparison against H1b_v1.

## Sections
| Section | Content | GPU |
|---|---|---|
| 1 | Setup (train pool, frozen-system counts from B3S Step 1) | No |
| 2 | UM diagnostic: per-mask calibration, image counts | No |
| 3 | H1b-K3 | No |
| 4 | Targets and comparison | No |

# 1. Setup

In [1]:
# 1.0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 1.1 — Paths, repo modules, train pool, frozen-system counts
# ============================================================

import importlib, json, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
FSC147_RAW = PROJECT_ROOT / "datasets" / "FSC147" / "raw"
TD_ROOT = PROJECT_ROOT / "cache" / "FSC147_train_dev400"
B3S_DIR = RESULTS_ROOT / "B3S_Semantic_Improvement"
B3U_DIR = RESULTS_ROOT / "B3U_Unified_Multiplicity"
B3U_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import src.datasets as datasets_mod
import src.pipeline.io_cache as io_cache
import src.pipeline.sg as sg
import src.pipeline.gpv as gpv
import src.pipeline.multiplicity as K
import src.pipeline.hybrid as hybrid
for m in (datasets_mod, io_cache, sg, gpv, K, hybrid):
    importlib.reload(m)

STRATA = ["S1_0_20", "S2_20_40", "S3_40_60", "S4_60_80", "S5_80_95", "S6_95_100"]
W = {"S1_0_20": .20, "S2_20_40": .20, "S3_40_60": .20, "S4_60_80": .20, "S5_80_95": .15, "S6_95_100": .05}
TD_DIRS = {"b0_masks": TD_ROOT / "b0_masks", "a3c_masks": TD_ROOT / "a3c_masks",
           "depth": TD_ROOT / "depth", "m1_split_labels": TD_ROOT / "m1_split_labels"}
TD_TILED = TD_ROOT / "san_vocab_tiled"
SG_CFG = sg.SGConfig.from_frozen_json(RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "SG_v1.1_frozen_config.json")

td_df = pd.read_csv(TD_ROOT / "FSC147_train_dev400.csv")
td_res = pd.read_csv(B3S_DIR / "B3S_step1_per_image.csv")
assert len(td_df) == 400 and len(td_res) == 400
train_ds = datasets_mod.FSC147Dataset(FSC147_RAW, split="train", load_images=False)

def gt_points(di):
    (_, cat), = train_ds[int(di)]["categories"].items()
    return np.asarray(cat["points"], dtype=np.float64).reshape(-1, 2)

def owned_gt(masks, gt_xy):
    # GT points owned per mask (each GT owned by the smallest mask containing it)
    n = masks.shape[0]
    if n == 0 or len(gt_xy) == 0:
        return np.zeros(n, int), np.zeros(n, int), 0
    h, w = masks.shape[1:]
    px = np.clip(np.rint(gt_xy[:, 0]).astype(int), 0, w - 1)
    py = np.clip(np.rint(gt_xy[:, 1]).astype(int), 0, h - 1)
    hits = masks[:, py, px]
    areas = masks.reshape(n, -1).sum(axis=1)
    owns = np.zeros(n, int)
    uncovered = 0
    for g in range(hits.shape[1]):
        holders = np.flatnonzero(hits[:, g])
        if len(holders):
            owns[holders[np.argmin(areas[holders])]] += 1
        else:
            uncovered += 1
    return owns, hits.sum(axis=1), uncovered

print("Repo commit:", REPO_COMMIT, "| pool:", len(td_df))

Repo commit: 0a513a5814f5aaf6f16a45a7985c8417871dd99d | pool: 400


# 2. UM diagnostic

Per mask: target area aᵢ, the image's object size s, the UM estimate mᵢ, and (diagnosis only) the number of GT points the mask owns. Mask type for reporting: **background** (owns 0 GT, target share < 0.5), **fragment/extra** (owns 0 GT, target share ≥ 0.5), **single** (owns 1), **merged** (owns ≥ 2). Also: GT points inside no mask (missing objects — a generation limit UM cannot fix).

In [ ]:
# ============================================================
# Step 2.1 — UM per mask and per image (CPU, cached inputs)
# ============================================================

mask_rows, img_rows = [], []
t0 = time.time()
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="UM"):
    di = r.dataset_index
    stem = io_cache.image_stem(r.image_id)
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    b0m, b0meta = io_cache.load_b0_masks(p["b0_masks"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    g = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery" / f"{int(di):04d}_{stem}.npz")
    keep = gpv.nms_masks(g["masks"], g["scores"], 0.5)
    r4_m, r4_s, r4_src = gpv.build_r4(g["masks"][keep], g["scores"][keep], b0m, b0meta["predicted_iou"], 0.5)
    with np.load(TD_TILED / f"{stem}.npz") as v:
        amax = v["tiled_argmax"]
    tgt = amax == 0
    n = r4_m.shape[0]
    gt_xy = gt_points(di)
    if n == 0:
        img_rows.append({"dataset_index": di, "UM": 0, "s": np.nan, "s_source": "none",
                         "n_masks": 0, "gt_uncovered": len(gt_xy)})
        continue
    flat = r4_m.reshape(n, -1)
    area = flat.sum(axis=1).astype(float)
    a = (flat & tgt.ravel()[None]).sum(axis=1).astype(float)          # target area
    share = a / np.maximum(area, 1)
    n_cls = int(amax.max()) + 1
    dom_t = np.array([np.bincount(amax[m], minlength=n_cls).argmax() == 0 if m.any() else False for m in r4_m])
    d1 = gpv.d1_peaks(san["target_score"])
    merged = gpv.merged_mask_flags(r4_m, gpv.points_per_mask(r4_m, d1), "A")
    verified = dom_t & ~merged & (r4_s >= np.median(r4_s))
    if verified.sum() >= 3:
        s, s_src = float(np.median(a[verified])), "verified_single"
    elif dom_t.any():
        s, s_src = float(np.median(a[dom_t])), "target_dominant"
    else:
        s, s_src = float(np.median(a)), "all_masks"
    s = max(s, 1.0)
    m_hat = np.rint(a / s).astype(int)
    owns, gt_in, uncovered = owned_gt(r4_m, gt_xy)
    for j in range(n):
        if owns[j] == 0:
            mtype = "background" if share[j] < 0.5 else "fragment/extra"
        elif owns[j] == 1:
            mtype = "single"
        else:
            mtype = "merged"
        mask_rows.append({"dataset_index": di, "density_stratum": r.density_stratum, "source": r4_src[j],
                          "mask_type": mtype, "owned_gt": int(owns[j]), "m_hat": int(m_hat[j]),
                          "a_over_s": a[j] / s, "target_share": share[j], "rule_A_merged": bool(merged[j])})
    img_rows.append({"dataset_index": di, "UM": int(m_hat.sum()), "s": s, "s_source": s_src,
                     "n_masks": n, "gt_uncovered": uncovered})

um_masks = pd.DataFrame(mask_rows)
um_img = (pd.DataFrame(img_rows)
          .merge(td_res[["dataset_index", "gt_count", "density_stratum", "B0", "GPV_R4_v1", "H1b_v1"]],
                 on="dataset_index", validate="one_to_one"))
assert (um_img.loc[um_img["n_masks"] > 0, "n_masks"].values
        == um_img.loc[um_img["n_masks"] > 0, "GPV_R4_v1"].values).all(), "R4 rebuild mismatch"
print(f"Runtime: {(time.time() - t0) / 60:.1f} min | masks: {len(um_masks)}")
print("Object-size source per image:")
display(um_img.groupby("density_stratum")["s_source"].value_counts().unstack(fill_value=0).reindex(STRATA))

UM:   0%|          | 0/400 [00:00<?, ?it/s]

Runtime: 16.6 min | masks: 19303
Object-size source per image:


s_source,all_masks,target_dominant,verified_single
density_stratum,,,
S1_0_20,0,8,72
S2_20_40,0,9,71
S3_40_60,0,4,56
S4_60_80,1,5,54
S5_80_95,1,4,55
S6_95_100,4,6,50


In [ ]:
# ============================================================
# Step 2.2 — Per-mask calibration: does m_hat match the GT each mask owns?
# ============================================================

def region(st):
    return "sparse S1-S2" if st in ("S1_0_20", "S2_20_40") else ("S3" if st == "S3_40_60" else "dense S4-S6")
um_masks["region"] = um_masks["density_stratum"].map(region)

cal = (um_masks.groupby(["region", "mask_type"])
       .agg(n=("m_hat", "size"), mean_owned=("owned_gt", "mean"), mean_m_hat=("m_hat", "mean"),
            share_m0=("m_hat", lambda x: (x == 0).mean()), share_m1=("m_hat", lambda x: (x == 1).mean()),
            share_m2plus=("m_hat", lambda x: (x >= 2).mean())))
print("Calibration by mask type (want: background & fragment -> m 0; single -> 1; merged -> owned):")
display(cal.round(3))

err_parts = (um_masks.assign(d=um_masks["m_hat"] - um_masks["owned_gt"])
             .groupby(["region", "mask_type"])["d"].sum()
             .unstack(fill_value=0))
per_img = um_masks.groupby("region")["dataset_index"].nunique()
print("\nSigned count contribution per image (m_hat - owned GT), by mask type:")
display(err_parts.div(per_img, axis=0).round(2))

miss = um_img.assign(region=um_img["density_stratum"].map(region)).groupby("region").apply(
    lambda d: pd.Series({"gt_per_img": d["gt_count"].mean(), "missing_per_img": d["gt_uncovered"].mean(),
                         "missing_share": d["gt_uncovered"].sum() / d["gt_count"].sum()}))
print("\nObjects with no mask at all (generation limit — UM cannot recover these):")
display(miss.round(2))

Calibration by mask type (want: background & fragment -> m 0; single -> 1; merged -> owned):


n  mean_owned  mean_m_hat  share_m0  \
region       mask_type                                                 
S3           background        168       0.000       0.982     0.887   
             fragment/extra    506       0.000       2.660     0.542   
             merged             35       5.457      28.914     0.000   
             single           1282       1.000       1.057     0.158   
dense S4-S6  background        777       0.000       1.727     0.892   
             fragment/extra   1084       0.000       2.977     0.477   
             merged            709      26.513      26.767     0.157   
             single          11452       1.000       0.898     0.199   
sparse S1-S2 background        627       0.000       0.536     0.927   
             fragment/extra    817       0.000       3.031     0.409   
             merged             39       4.744      15.308     0.051   
             single           1807       1.000       1.445     0.116   

                             share_m1  share_m2plus  
region       mask_type                               
S3           background         0.065         0.048  
             fragment/extra     0.292         0.166  
             merged             0.371         0.629  
             single             0.775         0.067  
dense S4-S6  background         0.049         0.059  
             fragment/extra     0.340         0.183  
             merged             0.425         0.419  
             single             0.752         0.049  
sparse S1-S2 background         0.030         0.043  
             fragment/extra     0.357         0.234  
             merged             0.359         0.590  
             single             0.780         0.104


Signed count contribution per image (m_hat - owned GT), by mask type:


mask_type,background,fragment/extra,merged,single
region,,,,
S3,2.75,22.43,13.68,1.22
dense S4-S6,7.46,17.93,1.00,-6.49
sparse S1-S2,2.10,15.48,2.58,5.03



Objects with no mask at all (generation limit — UM cannot recover these):


/tmp/ipykernel_1116/3776111446.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  miss = um_img.assign(region=um_img["density_stratum"].map(region)).groupby("region").apply(


,gt_per_img,missing_per_img,missing_share
region,,,
S3,27.37,2.82,0.10
dense S4-S6,190.19,22.14,0.12
sparse S1-S2,13.38,0.92,0.07


# 3. H1b-K3

In [ ]:
# ============================================================
# Step 3.1 — H1b-K3 (CPU, cached inputs)
# ============================================================

def h1b_k3_count(r4_m, d1, a3c, depth):
    base, info = hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)
    if not info["stage2_on"] or r4_m.shape[0] == 0:
        return base, False
    merged = gpv.merged_mask_flags(r4_m, gpv.points_per_mask(r4_m, d1), "A")
    if not merged.any():
        return base, False
    anchor = r4_m[~merged]
    areas = anchor.reshape(anchor.shape[0], -1).sum(axis=1) if anchor.shape[0] else np.zeros(0)
    a_ref = float(np.median(areas)) if areas.size else float(np.median(r4_m[merged].reshape(merged.sum(), -1).sum(1)))
    taken = anchor.any(axis=0) if anchor.shape[0] else np.zeros(r4_m.shape[1:], bool)
    total = int(anchor.shape[0])
    for i in np.flatnonzero(merged):
        region_i = r4_m[i] & ~taken
        taken |= region_i
        ar = int(region_i.sum())
        if ar > 0 and ar >= 0.5 * a_ref:
            total += max(1, K.k3_watershed_depth_gradient(region_i, depth, 0.25 * a_ref))
    return float(total), True

k3_rows = []
t0 = time.time()
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="H1b-K3"):
    di = r.dataset_index
    stem = io_cache.image_stem(r.image_id)
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    b0m, b0meta = io_cache.load_b0_masks(p["b0_masks"])
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    g = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery" / f"{int(di):04d}_{stem}.npz")
    keep = gpv.nms_masks(g["masks"], g["scores"], 0.5)
    r4_m, _, _ = gpv.build_r4(g["masks"][keep], g["scores"][keep], b0m, b0meta["predicted_iou"], 0.5)
    d1 = gpv.d1_peaks(san["target_score"])
    cnt, used = h1b_k3_count(r4_m, d1, a3c, dep)
    k3_rows.append({"dataset_index": di, "H1b-K3": cnt, "k3_used": used})
k3_df = pd.DataFrame(k3_rows)
um_img = um_img.merge(k3_df, on="dataset_index", validate="one_to_one")
off = ~um_img["k3_used"]
print(f"Runtime: {(time.time() - t0) / 60:.1f} min")
print("Gate-OFF images equal H1b_v1:", int(np.isclose(um_img.loc[off, "H1b-K3"], um_img.loc[off, "H1b_v1"]).sum()),
      "/", int(off.sum()))
print("K3 used, by stratum:")
display(um_img.groupby("density_stratum")["k3_used"].agg(["sum", "size"]).reindex(STRATA))

H1b-K3:   0%|          | 0/400 [00:00<?, ?it/s]

Runtime: 6.9 min
Gate-OFF images equal H1b_v1: 365 / 365
K3 used, by stratum:


,sum,size
density_stratum,,
S1_0_20,4,80
S2_20_40,5,80
S3_40_60,2,60
S4_60_80,3,60
S5_80_95,9,60
S6_95_100,12,60


# 4. Targets and comparison

In [ ]:
# ============================================================
# Step 4.1 — Per-stratum metrics and targets T1, T2', T3'
# ============================================================

SYS = ["B0", "GPV_R4_v1", "H1b_v1", "UM", "H1b-K3"]
gt = um_img["gt_count"].to_numpy(float); st = um_img["density_stratum"].to_numpy()
AE = {s: np.abs(um_img[s].to_numpy(float) - gt) for s in SYS}
IDX = {}
for j, k in enumerate(STRATA):
    pos = np.flatnonzero(st == k)
    IDX[k] = pos[np.random.default_rng(1000 + j).integers(0, len(pos), size=(10_000, len(pos)))]

def sci(a, b, k):
    d = AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)
    return AE[a][st == k].mean() - AE[b][st == k].mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

def rci(a, b):
    d = sum(w * (AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)) for k, w in W.items())
    o = sum(w * (AE[a][st == k].mean() - AE[b][st == k].mean()) for k, w in W.items())
    return o, np.percentile(d, 2.5), np.percentile(d, 97.5)

res = pd.DataFrame([{"system": s,
                     "MAE_reweighted": sum(w * AE[s][st == k].mean() for k, w in W.items()),
                     "mean_signed": float((um_img[s] - um_img["gt_count"]).mean()),
                     **{f"MAE_{k[:2]}": AE[s][st == k].mean() for k in STRATA}} for s in SYS])
display(res.round(2))

rows = []
for s in ["UM", "H1b-K3"]:
    rec = {"system": s}; t1 = []; t2 = []
    for k in ["S1_0_20", "S2_20_40"]:
        d, lo, hi = sci(s, "B0", k); rec[f"{k[:2]} vs B0"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t1.append(lo <= 0)
    for k in ["S5_80_95", "S6_95_100"]:
        d, lo, hi = sci(s, "H1b_v1", k); rec[f"{k[:2]} vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t2.append(lo <= 0)
    d, lo, hi = rci(s, "H1b_v1"); rec["overall vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"
    rec.update({"T1": all(t1), "T2'": all(t2), "T3'": hi < 0, "ALL": all(t1) and all(t2) and hi < 0})
    rows.append(rec)
targets = pd.DataFrame(rows)
print("\nTargets (difference [95% CI]; negative = system better):")
display(targets)

um_masks.to_csv(B3U_DIR / "B3U_um_masks.csv", index=False)
um_img.to_csv(B3U_DIR / "B3U_per_image.csv", index=False)
res.to_csv(B3U_DIR / "B3U_metrics.csv", index=False)
targets.to_csv(B3U_DIR / "B3U_targets.csv", index=False)
with open(B3U_DIR / "B3U_metadata.json", "w") as f:
    json.dump({"notebook": "B3U_Unified_Multiplicity", "stage": "UM diagnostic + H1b-K3 (train_dev400)",
               "UM": {"masks": "GPV_R4 kept masks", "a_i": "target-winning pixels, tiled expanded-vocabulary map",
                      "s": "median a of verified singles (target-dominant, not rule-A merged, score >= image median); "
                           "fallbacks: target-dominant, then all masks; >= 1 px",
                      "m_i": "round(a_i / s)"},
               "H1b-K3": "H1b; where H1b's SG stage-2 gate is ON and rule-A merged masks exist: anchor masks + "
                         "sum over qualifying merged regions of max(1, K3 segments) (GPV_K region definition)",
               "targets": {"T1": "S1,S2 not sig. worse than B0", "T2'": "S5,S6 not sig. worse than H1b_v1",
                           "T3'": "reweighted MAE sig. better than H1b_v1"},
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", B3U_DIR)

,system,MAE_reweighted,mean_signed,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0,35.67,-62.67,5.56,7.74,13.30,22.93,59.03,338.08
1,GPV_R4_v1,29.94,-46.78,8.62,7.70,12.45,16.13,40.52,297.63
2,H1b_v1,28.65,-41.28,10.46,9.49,12.35,14.20,37.33,274.98
3,UM,57.48,14.28,31.82,22.10,42.67,33.63,103.75,317.48
4,H1b-K3,37.32,-18.66,10.01,10.38,16.83,39.67,51.95,282.90



Targets (difference [95% CI]; negative = system better):


,system,S1 vs B0,S2 vs B0,S5 vs H1b,S6 vs H1b,overall vs H1b,T1,T2',T3',ALL
0,UM,"+26.26 [+11.41, +44.48]","+14.36 [+5.55, +24.92]","+66.42 [+18.98, +136.40]","+42.50 [-15.33, +106.58]","+28.83 [+18.61, +41.25]",False,False,False,False
1,H1b-K3,"+4.45 [+2.75, +6.20]","+2.64 [-0.03, +5.89]","+14.62 [-0.82, +38.35]","+7.92 [-14.93, +34.02]","+8.67 [+1.49, +19.46]",False,True,False,False


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3U_Unified_Multiplicity


## Sections 2–4 result

- **UM over-counts** (reweighted 57.48; mean signed +14.3). Almost all the excess comes from **container masks**: large target-looking masks owning no GT because the objects inside are also segmented individually (IoU-NMS keeps both). UM summed target area over **overlapping** masks — double counting.
- **Promising:** genuine merged masks in dense scenes are calibrated almost exactly (mean UM 26.8 vs 26.5 GT owned); background masks → 0 in 89–93%; singles → 1 in 75–78%.
- Objects with no mask at all: 7% (sparse), 12% (dense) — generation is not the main limit.
- **H1b-K3: negative** (reweighted 37.32; S4 14.20 → 39.67; S6 no gain). Gate-off images reproduce H1b_v1 exactly (365/365).

# 5. UM-v2 — exclusive target area

## Pre-declared fix (motivated by the Section 2 diagnosis; no new parameter)
- **Exclusive assignment:** every pixel belongs to **one** kept mask only — the **smallest** mask containing it (same rule as GT ownership).
- **Exclusive target area** eᵢ = target-winning pixels (tiled expanded-vocabulary map) among mask i's exclusive pixels.
- **Object size s** = median eᵢ of verified single masks (target-dominant, not rule-A merged, SAM2 score ≥ image median); fallbacks as in UM (target-dominant, then all masks; ≥ 1 px).
- **mᵢ = round(eᵢ / s)**; count = Σ mᵢ.

Everything else (mask set, vocabulary map, verified-single definition, targets T1, T2′, T3′) is unchanged from UM. If UM-v2 fails the targets, the UM idea is considered properly tested.

**Run order:** CPU → Section 1 → Section 5 (Sections 2–4 not needed).

In [3]:
# ============================================================
# Step 5.1 — UM-v2 per mask and per image (CPU, cached inputs)
# ============================================================

def exclusive_assignment(masks):
    # Pixel -> index of the SMALLEST mask containing it (-1 = none)
    n = masks.shape[0]
    areas = masks.reshape(n, -1).sum(axis=1)
    owner = np.full(masks.shape[1:], -1, dtype=np.int32)
    for i in np.argsort(-areas, kind="stable"):        # largest first, smaller masks overwrite
        owner[masks[i]] = i
    return owner

mask_rows2, img_rows2 = [], []
t0 = time.time()
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="UM-v2"):
    di = r.dataset_index
    stem = io_cache.image_stem(r.image_id)
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    b0m, b0meta = io_cache.load_b0_masks(p["b0_masks"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    g = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery" / f"{int(di):04d}_{stem}.npz")
    keep = gpv.nms_masks(g["masks"], g["scores"], 0.5)
    r4_m, r4_s, r4_src = gpv.build_r4(g["masks"][keep], g["scores"][keep], b0m, b0meta["predicted_iou"], 0.5)
    with np.load(TD_TILED / f"{stem}.npz") as v:
        amax = v["tiled_argmax"]
    tgt = amax == 0
    n = r4_m.shape[0]
    gt_xy = gt_points(di)
    if n == 0:
        img_rows2.append({"dataset_index": di, "UM-v2": 0, "s2": np.nan, "s2_source": "none"})
        continue
    owner = exclusive_assignment(r4_m)
    e = np.bincount(owner[tgt & (owner >= 0)].ravel(), minlength=n).astype(float)   # exclusive target area
    excl = np.bincount(owner[owner >= 0].ravel(), minlength=n).astype(float)       # exclusive area
    n_cls = int(amax.max()) + 1
    dom_t = np.array([np.bincount(amax[m], minlength=n_cls).argmax() == 0 if m.any() else False for m in r4_m])
    d1 = gpv.d1_peaks(san["target_score"])
    merged = gpv.merged_mask_flags(r4_m, gpv.points_per_mask(r4_m, d1), "A")
    verified = dom_t & ~merged & (r4_s >= np.median(r4_s))
    if verified.sum() >= 3:
        s, s_src = float(np.median(e[verified])), "verified_single"
    elif dom_t.any():
        s, s_src = float(np.median(e[dom_t])), "target_dominant"
    else:
        s, s_src = float(np.median(e)), "all_masks"
    s = max(s, 1.0)
    m_hat = np.rint(e / s).astype(int)
    owns, gt_in, _ = owned_gt(r4_m, gt_xy)
    full_area = r4_m.reshape(n, -1).sum(axis=1).astype(float)
    for j in range(n):
        if owns[j] == 0:
            mtype = "container" if gt_in[j] > 0 else ("background" if e[j] < 0.5 * max(excl[j], 1) else "fragment/extra")
        elif owns[j] == 1:
            mtype = "single"
        else:
            mtype = "merged"
        mask_rows2.append({"dataset_index": di, "density_stratum": r.density_stratum, "source": r4_src[j],
                           "mask_type": mtype, "owned_gt": int(owns[j]), "m_hat": int(m_hat[j]),
                           "e_over_s": e[j] / s, "exclusive_share_of_mask": excl[j] / max(full_area[j], 1)})
    img_rows2.append({"dataset_index": di, "UM-v2": int(m_hat.sum()), "s2": s, "s2_source": s_src})

um2_masks = pd.DataFrame(mask_rows2)
um2_img = (pd.DataFrame(img_rows2)
           .merge(td_res[["dataset_index", "gt_count", "density_stratum", "B0", "GPV_R4_v1", "H1b_v1"]],
                  on="dataset_index", validate="one_to_one"))
print(f"Runtime: {(time.time() - t0) / 60:.1f} min | masks: {len(um2_masks)}")

UM-v2:   0%|          | 0/400 [00:00<?, ?it/s]

Runtime: 17.4 min | masks: 19303


In [4]:
# ============================================================
# Step 5.2 — Calibration of UM-v2 by mask type
# ============================================================
#   container       : owns 0 GT but contains GT owned by smaller masks (double-count risk)
#   background      : owns 0 GT, contains none, exclusive target share < 0.5
#   fragment/extra  : owns 0 GT, contains none, exclusive target share >= 0.5

def region(st):
    return "sparse S1-S2" if st in ("S1_0_20", "S2_20_40") else ("S3" if st == "S3_40_60" else "dense S4-S6")
um2_masks["region"] = um2_masks["density_stratum"].map(region)

cal2 = (um2_masks.groupby(["region", "mask_type"])
        .agg(n=("m_hat", "size"), mean_owned=("owned_gt", "mean"), mean_m_hat=("m_hat", "mean"),
             share_m0=("m_hat", lambda x: (x == 0).mean()), share_m1=("m_hat", lambda x: (x == 1).mean()),
             share_m2plus=("m_hat", lambda x: (x >= 2).mean())))
print("UM-v2 calibration by mask type:")
display(cal2.round(3))

per_img2 = um2_masks.groupby("region")["dataset_index"].nunique()
contrib2 = (um2_masks.assign(d=um2_masks["m_hat"] - um2_masks["owned_gt"])
            .groupby(["region", "mask_type"])["d"].sum().unstack(fill_value=0))
print("\nSigned count contribution per image (m_hat - owned GT), by mask type:")
display(contrib2.div(per_img2, axis=0).round(2))

UM-v2 calibration by mask type:


n  mean_owned  mean_m_hat  share_m0  \
region       mask_type                                                 
S3           background        165       0.000       1.036     0.903   
             container         101       0.000       1.851     0.158   
             fragment/extra    408       0.000       1.995     0.662   
             merged             35       5.457      25.114     0.000   
             single           1282       1.000       0.973     0.153   
dense S4-S6  background        757       0.000       1.004     0.909   
             container         124       0.000       7.347     0.161   
             fragment/extra    980       0.000       1.648     0.526   
             merged            709      26.513      16.791     0.159   
             single          11452       1.000       0.883     0.199   
sparse S1-S2 background        617       0.000       0.462     0.932   
             container         125       0.000       3.776     0.240   
             fragment/extra    702       0.000       1.969     0.470   
             merged             39       4.744      10.154     0.051   
             single           1807       1.000       1.283     0.111   

                             share_m1  share_m2plus  
region       mask_type                               
S3           background         0.036         0.061  
             container          0.614         0.228  
             fragment/extra     0.245         0.093  
             merged             0.429         0.571  
             single             0.789         0.059  
dense S4-S6  background         0.049         0.042  
             container          0.387         0.452  
             fragment/extra     0.346         0.129  
             merged             0.420         0.420  
             single             0.751         0.049  
sparse S1-S2 background         0.028         0.041  
             container          0.480         0.280  
             fragment/extra     0.328         0.202  
             merged             0.333         0.615  
             single             0.790         0.099


Signed count contribution per image (m_hat - owned GT), by mask type:


mask_type,background,container,fragment/extra,merged,single
region,,,,,
S3,2.85,3.12,13.57,11.47,-0.58
dense S4-S6,4.22,5.06,8.97,-38.29,-7.47
sparse S1-S2,1.78,2.95,8.64,1.32,3.20


In [5]:
# ============================================================
# Step 5.3 — Metrics and targets (UM-v2 vs references and UM)
# ============================================================

cmp = um2_img.merge(um_img[["dataset_index", "UM"]], on="dataset_index", how="left") \
    if "um_img" in globals() else um2_img
SYS2 = ["B0", "GPV_R4_v1", "H1b_v1"] + (["UM"] if "UM" in cmp.columns else []) + ["UM-v2"]
gt2 = cmp["gt_count"].to_numpy(float); st2 = cmp["density_stratum"].to_numpy()
AE2 = {s: np.abs(cmp[s].to_numpy(float) - gt2) for s in SYS2}
IDX2 = {}
for j, k in enumerate(STRATA):
    pos = np.flatnonzero(st2 == k)
    IDX2[k] = pos[np.random.default_rng(1000 + j).integers(0, len(pos), size=(10_000, len(pos)))]

def sci2(a, b, k):
    d = AE2[a][IDX2[k]].mean(1) - AE2[b][IDX2[k]].mean(1)
    return AE2[a][st2 == k].mean() - AE2[b][st2 == k].mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

def rci2(a, b):
    d = sum(w * (AE2[a][IDX2[k]].mean(1) - AE2[b][IDX2[k]].mean(1)) for k, w in W.items())
    o = sum(w * (AE2[a][st2 == k].mean() - AE2[b][st2 == k].mean()) for k, w in W.items())
    return o, np.percentile(d, 2.5), np.percentile(d, 97.5)

res2 = pd.DataFrame([{"system": s,
                      "MAE_reweighted": sum(w * AE2[s][st2 == k].mean() for k, w in W.items()),
                      "mean_signed": float((cmp[s] - cmp["gt_count"]).mean()),
                      **{f"MAE_{k[:2]}": AE2[s][st2 == k].mean() for k in STRATA}} for s in SYS2])
display(res2.round(2))

rec = {"system": "UM-v2"}; t1 = []; t2 = []
for k in ["S1_0_20", "S2_20_40"]:
    d, lo, hi = sci2("UM-v2", "B0", k); rec[f"{k[:2]} vs B0"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t1.append(lo <= 0)
for k in ["S5_80_95", "S6_95_100"]:
    d, lo, hi = sci2("UM-v2", "H1b_v1", k); rec[f"{k[:2]} vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t2.append(lo <= 0)
d, lo, hi = rci2("UM-v2", "H1b_v1"); rec["overall vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"
rec.update({"T1": all(t1), "T2'": all(t2), "T3'": hi < 0, "ALL": all(t1) and all(t2) and hi < 0})
targets2 = pd.DataFrame([rec])
print("\nTargets (difference [95% CI]; negative = UM-v2 better):")
display(targets2)

D5 = B3U_DIR / "section5_um_v2"
D5.mkdir(parents=True, exist_ok=True)
um2_masks.to_csv(D5 / "B3U_s5_um2_masks.csv", index=False)
cmp.to_csv(D5 / "B3U_s5_per_image.csv", index=False)
res2.to_csv(D5 / "B3U_s5_metrics.csv", index=False)
targets2.to_csv(D5 / "B3U_s5_targets.csv", index=False)
with open(D5 / "B3U_s5_metadata.json", "w") as f:
    json.dump({"section": "5 — UM-v2 (exclusive target area)",
               "rule": "pixel owned by the smallest kept mask containing it; m_i = round(exclusive target area / s)",
               "s": "median exclusive target area of verified singles; fallbacks as UM",
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", D5)

,system,MAE_reweighted,mean_signed,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0,35.67,-62.67,5.56,7.74,13.30,22.93,59.03,338.08
1,GPV_R4_v1,29.94,-46.78,8.62,7.70,12.45,16.13,40.52,297.63
2,H1b_v1,28.65,-41.28,10.46,9.49,12.35,14.20,37.33,274.98
3,UM-v2,42.63,-11.42,22.76,16.99,33.73,28.40,64.02,252.95



Targets (difference [95% CI]; negative = UM-v2 better):


,system,S1 vs B0,S2 vs B0,S5 vs H1b,S6 vs H1b,overall vs H1b,T1,T2',T3',ALL
0,UM-v2,"+17.20 [+7.69, +28.83]","+9.25 [+2.93, +16.64]","+26.68 [+2.90, +57.67]","-22.03 [-58.58, +14.95]","+13.98 [+7.61, +21.10]",False,False,False,False


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3U_Unified_Multiplicity/section5_um_v2


# B3U conclusion

**Candidates tested on train_dev400 (pre-declared; targets T1, T2′, T3′):**

| | Reweighted | Mean signed | S1 | S2 | S5 | S6 | Targets |
|---|---|---|---|---|---|---|---|
| H1b_v1 (reference) | 28.65 | −41.3 | 10.46 | 9.49 | 37.33 | 274.98 | — |
| UM | 57.48 | +14.3 | 31.82 | 22.10 | 103.75 | 317.48 | none |
| UM-v2 (exclusive area) | 42.63 | −11.4 | 22.76 | 16.99 | 64.02 | 252.95 | none |
| H1b-K3 | 37.32 | −18.7 | 10.01 | 10.38 | 51.95 | 282.90 | T2′ only |

**Findings:**
- **UM over-counted through overlapping masks:** container masks (a group mask over objects also segmented individually) were counted by area on top of their contents.
- **UM-v2 fixed part of it,** but containers keep the gaps between objects, which look like target in SAN's blurred maps; target-looking fragments still count as ~2 objects; and genuine merged masks become under-estimated once smaller masks claim their pixels.
- **Area-based counting amplifies evidence errors:** a mask counted as 1 is wrong by at most 1, but a mask counted by area is wrong by (area error ÷ object size). Mask counting is more robust wherever SAM2 separates objects.
- **But area carries real dense-scene information:** UM's genuine merged masks in dense scenes were calibrated almost exactly (26.8 vs 26.5 GT owned), and UM-v2 has the best S6 of any system so far (252.95, not significant).
- **H1b-K3 is negative:** K3 on H1b's merged regions over-splits medium-density scenes (S4 14.20 → 39.67) without improving S6. Gate-off images reproduce H1b_v1 exactly (365/365).
- **Missing objects (no mask at all)** are 7% of GT in sparse scenes and 12% in dense: generation is not the main limit on recall.

**Conclusion:** the remaining ceiling lies less in how evidence is combined than in the quality of the masks and maps themselves (fragments, containers, target-looking gaps, merged groups). The next direction should improve mask and map quality directly.